# Q1 — Databricks Runtime vs Spark Local

**Semana 2 | Spark no Databricks**

O Databricks Runtime (DBR) não é o Spark open-source puro. Ele inclui otimizações exclusivas (Photon, AQE aprimorado, Delta Engine) e configurações padrão diferentes.

> ⚙️ **Este notebook foi validado com Serverless compute.**

### O que o Serverless revela sobre o DBR

Ao tentar inspecionar `spark.databricks.clusterUsageTags.sparkVersion` no Serverless, você recebe `CONFIG_NOT_AVAILABLE`. Isso **não é um bug** — é o comportamento esperado e traz um aprendizado importante:

| | All-Purpose Cluster | Serverless |
|---|---|---|
| Versão do DBR | Você escolhe (ex: 15.4 LTS) | Databricks gerencia automaticamente |
| `clusterUsageTags.*` | Disponível | ❌ Não disponível |
| `spark.databricks.runtime.version` | Disponível | ❌ Não disponível |
| Configs de AQE e Delta | Disponível | ✅ Disponível |
| `spark.sql.shuffle.partitions` | Disponível | ✅ Disponível |

**No Serverless, você não gerencia a versão do runtime** — a Databricks roda sempre a última versão estável. A abstração é total.

O que faremos neste Q1: comparar as configs que **são** acessíveis no Serverless com os defaults do Spark open-source — que é onde está a diferença que importa para o seu código.

---
## Parte 1 — Configurações do Databricks Runtime

Vamos inspecionar as configs do Spark que o DBR define automaticamente.
Preste atenção em `spark.sql.shuffle.partitions` — o padrão do Spark open-source é **200**.

In [ ]:
import sys

# spark.databricks.clusterUsageTags.sparkVersion só existe em clusters All-Purpose
# No Serverless, usar spark.databricks.runtime.version (ou o fallback abaixo)
def get_conf_safe(key, fallback="(não disponível no Serverless)"):
    try:
        return spark.conf.get(key)
    except Exception:
        return fallback

dbr_version = (
    get_conf_safe("spark.databricks.runtime.version")
    or get_conf_safe("spark.databricks.clusterUsageTags.sparkVersion")
)

print(f"Databricks Runtime : {dbr_version}")
print(f"Spark version      : {spark.version}")
print(f"Python version     : {sys.version.split()[0]}")
print(f"Compute type       : Serverless (clusterUsageTags não disponível)")

In [ ]:
# Inspecionar configurações críticas do DBR
# Compare os valores com os defaults do Spark open-source documentados abaixo
configs = {
    # AQE — Adaptive Query Execution (default open-source: false antes do Spark 3.2)
    "spark.sql.adaptive.enabled": "open-source default: true (Spark 3.2+)",
    # AQE: coalesce automático de partições após shuffle
    "spark.sql.adaptive.coalescePartitions.enabled": "open-source default: true",
    # Shuffle partitions — open-source: 200 (fixo); DBR: auto via AQE
    "spark.sql.shuffle.partitions": "open-source default: 200",
    # Delta optimizeWrite — exclusivo do DBR, não existe no Spark open-source
    "spark.databricks.delta.optimizeWrite.enabled": "exclusivo DBR, não existe no OSS",
    # Delta auto compact — exclusivo do DBR
    "spark.databricks.delta.autoCompact.enabled": "exclusivo DBR, não existe no OSS",
}

print(f"{'Configuração':<55} {'Valor no DBR':<30} {'Referência'}")
print("-" * 120)
for config, referencia in configs.items():
    try:
        valor = spark.conf.get(config)
    except Exception:
        valor = "(não definida / usa default interno)"
    print(f"{config:<55} {valor:<30} {referencia}")

In [ ]:
# Listar configurações relevantes do DBR via spark.sql("SET -v")
# spark.sparkContext não é suportado no Serverless (JVM_ATTRIBUTE_NOT_SUPPORTED)
# Alternativa: "SET -v" retorna todas as configs como DataFrame

prefixos_interesse = ("spark.sql.adaptive", "spark.databricks.delta", "spark.databricks.cluster")

df_configs = (
    spark.sql("SET -v")
    .filter(
        " OR ".join([f"key LIKE '{p}%'" for p in prefixos_interesse])
    )
    .select("key", "value")
    .orderBy("key")
)

print(f"Configurações encontradas: {df_configs.count()}\n")
display(df_configs)

---
## Parte 2 — O que muda com AQE no shuffle.partitions?

No Spark open-source puro, `spark.sql.shuffle.partitions = 200` é um valor **fixo** que se aplica a toda operação de shuffle (`groupBy`, `join`, `orderBy`).

No DBR com AQE habilitado, esse valor vira um **máximo**: o AQE monitora os dados reais e coalesce as partições de shuffle automaticamente.

**Consequência prática para Serverless:**
- Um cluster single-node com 200 partições de shuffle em dados pequenos é pura overhead
- Com AQE, o DBR reduz automaticamente para o número ideal
- Você ainda pode forçar manualmente: `spark.conf.set("spark.sql.shuffle.partitions", "8")`

In [ ]:
# Demonstrar o impacto de shuffle.partitions com um groupBy real
# NOTA: .rdd.getNumPartitions() não funciona no Serverless (exige JVM/SparkContext)
# Para observar o efeito do AQE, usamos spark.sql("SET spark.sql.shuffle.partitions")
from pyspark.sql import functions as F

df_demo = spark.range(1_000_000).withColumn("grupo", (F.col("id") % 50).cast("string"))

# Com 200 partitions (default open-source)
spark.conf.set("spark.sql.shuffle.partitions", "200")
print(f"shuffle.partitions configurado para: {spark.conf.get('spark.sql.shuffle.partitions')}")
df_demo.groupBy("grupo").count().count()  # força execução do shuffle
print("  → groupBy executado com 200 partições de shuffle\n")

# Com 8 partitions (mais adequado para Serverless/single-node)
spark.conf.set("spark.sql.shuffle.partitions", "8")
print(f"shuffle.partitions configurado para: {spark.conf.get('spark.sql.shuffle.partitions')}")
df_demo.groupBy("grupo").count().count()  # força execução do shuffle
print("  → groupBy executado com 8 partições de shuffle\n")

# Restaurar para auto (deixar AQE decidir)
spark.conf.set("spark.sql.shuffle.partitions", "auto")
print(f"Restaurado para: {spark.conf.get('spark.sql.shuffle.partitions')} (AQE decide)")
print("\n💡 No Serverless, .rdd.getNumPartitions() não está disponível.")
print("   O controle de particionamento interno é gerenciado automaticamente pelo AQE.")

---
## Resumo Q1 — Diferenças DBR vs Spark Open-Source

| Configuração | Spark OSS | Databricks Runtime | Impacto |
|---|---|---|---|
| `shuffle.partitions` | 200 (fixo) | `auto` via AQE | Melhor performance em dados pequenos |
| `delta.optimizeWrite` | não existe | `true` | Compacta arquivos pequenos automaticamente |
| `delta.autoCompact` | não existe | `true` | Reduz número de arquivos Delta |
| AQE habilitado | `true` (Spark 3.2+) | `true` + extensões | Planos de execução adaptativos |
| `display()` | não existe | função built-in | Visualização rica de DataFrames |
| `dbutils` | não existe | built-in | Filesystem, secrets, widgets |
| Photon Engine | não existe | disponível | Execução vetorizada em C++ |

> 💡 **Para a comparação com Databricks Connect local**, execute `Q4_taxi_local.py` no VS Code.
> A grande diferença: no VS Code, `display()` não existe — use `.show()` ou `.toPandas()`.